# 03 - Explore LANL authentication telemetry

This notebook complements `ml.prepare_login_data` and `ml.train_login_anomaly`; it does not replace them. The raw `auth.txt` is approximately 73 GB, so the raw-data cells below read only a small prefix. That prefix is useful for checking the schema, but is not a representative sample of the full dataset. The optional processed-feature cell scans the generated Parquet file instead.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LANL_DIR = ROOT / "data" / "raw" / "lanl"
AUTH_PATH = LANL_DIR / "auth.txt"
REDTEAM_PATH = LANL_DIR / "redteam.txt"
FEATURE_PATH = ROOT / "data" / "processed" / "lanl_login_features.parquet"
MANIFEST_PATH = ROOT / "data" / "processed" / "lanl_login_features_manifest.json"

for path in (AUTH_PATH, REDTEAM_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required LANL input file was not found: {path}")

print(f"Repository: {ROOT}")
for path in (AUTH_PATH, REDTEAM_PATH):
    print(f"{path.name}: {path.stat().st_size / (1024 ** 3):.2f} GiB")
print(f"Processed features available: {FEATURE_PATH.is_file()}")
print(f"Preprocessing manifest available: {MANIFEST_PATH.is_file()}")

## 1. Check the raw record formats

Both files are headerless and whitespace-free, comma-separated records. This reads only the first three lines from each file.

In [ ]:
from itertools import islice

for path in (AUTH_PATH, REDTEAM_PATH):
    print(f"\n{path.name}")
    with path.open("r", encoding="utf-8") as file:
        for line in islice(file, 3):
            print(line.rstrip())

## 2. Inspect the red-team labels

Each label is an exact authentication-event key: epoch, source user, source computer, and destination computer. These are compromise labels, not exhaustive ground truth for all malicious activity.

In [ ]:
redteam_columns = ["epoch", "source_user", "source_computer", "destination_computer"]
redteam = pd.read_csv(
    REDTEAM_PATH,
    header=None,
    names=redteam_columns,
    dtype={
        "epoch": "int64",
        "source_user": "string",
        "source_computer": "string",
        "destination_computer": "string",
    },
    keep_default_na=False,
)

print(f"Rows: {len(redteam):,}")
print(f"Unique exact event keys: {redteam.drop_duplicates().shape[0]:,}")
print(f"Epoch range: {redteam['epoch'].min():,} to {redteam['epoch'].max():,}")
print("\nMost common source users in the label file:")
display(redteam["source_user"].value_counts().head(10).rename("label_rows"))
display(redteam.head(10))

## 3. Preview a bounded prefix of authentication events

This cell reads at most 100,000 rows from the beginning of `auth.txt`; it does not scan the 73 GB file. Counts and epoch ranges here describe only that prefix and must not be reported as full-dataset statistics.

In [ ]:
auth_columns = [
    "epoch", "source_user", "destination_user", "source_computer",
    "destination_computer", "auth_type", "logon_type", "orientation", "outcome",
]
auth_prefix = pd.read_csv(
    AUTH_PATH,
    header=None,
    names=auth_columns,
    nrows=100_000,
    dtype={"epoch": "int64"},
    keep_default_na=False,
)

print(f"Rows previewed: {len(auth_prefix):,}")
print(f"Prefix epoch range: {auth_prefix['epoch'].min():,} to {auth_prefix['epoch'].max():,}")
for column in ("outcome", "auth_type", "logon_type", "orientation"):
    print(f"\n{column} values in this prefix:")
    display(auth_prefix[column].value_counts(dropna=False).head(15))
display(auth_prefix.head())

## 4. Plot the label timeline

LANL timestamps are relative epochs. The horizontal axis below uses relative hours, not calendar dates or local wall-clock time.

In [ ]:
redteam_timeline = redteam.assign(hour_index=redteam["epoch"] // 3600)
labels_by_hour = redteam_timeline.groupby("hour_index").size()

ax = labels_by_hour.plot(figsize=(12, 4), linewidth=1)
ax.set(title="LANL red-team label records over relative time", xlabel="Relative hour index", ylabel="Label records")
plt.tight_layout()
plt.show()

print(f"Label records: {len(redteam):,}")
print(f"Hours containing at least one label record: {len(labels_by_hour):,}")

## 5. Review the full preprocessing audit, if available

The manifest is generated by `python -m ml.prepare_login_data`. It contains full-stream counts and exact label-match coverage; unlike the bounded raw preview above, those values summarize the complete authentication file.

In [ ]:
if MANIFEST_PATH.is_file():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    audit_keys = [
        "auth_rows_validated",
        "auth_events_aggregated",
        "auth_events_without_source_user",
        "redteam_rows",
        "unique_redteam_labels",
        "matched_redteam_auth_events",
        "matched_unique_redteam_labels",
        "unmatched_unique_redteam_labels",
        "user_hour_rows",
        "redteam_labeled_user_hours",
        "label_match_key",
    ]
    display(pd.Series({key: manifest.get(key) for key in audit_keys}, name="full preprocessing audit"))
else:
    print("Manifest not found. Run `python -m ml.prepare_login_data` from the repository root to create it.")

## 6. Optional: inspect generated user-hour features

This cell uses DuckDB to aggregate the processed Parquet file without loading all feature rows into pandas. It scans the generated file, which is much smaller than `auth.txt`, but can still take time. Skip it if the preprocessing output is not available.

In [ ]:
if FEATURE_PATH.is_file():
    import duckdb

    parquet_path_sql = str(FEATURE_PATH).replace("\\", "/").replace("'", "''")
    with duckdb.connect(":memory:") as con:
        by_day = con.execute(
            f"""
            SELECT hour_index // 24 AS relative_day_index,
                   COUNT(*) AS user_hour_rows,
                   SUM(event_count) AS auth_events,
                   SUM(redteam_event_count) AS matched_redteam_events,
                   COUNT(*) FILTER (WHERE redteam_event_count > 0) AS labeled_user_hours
            FROM read_parquet('{parquet_path_sql}')
            GROUP BY relative_day_index
            ORDER BY relative_day_index
            """
        ).df()

    print(f"User-hour rows: {by_day['user_hour_rows'].sum():,.0f}")
    print(f"Aggregated auth events: {by_day['auth_events'].sum():,.0f}")
    print(f"Matched red-team events: {by_day['matched_redteam_events'].sum():,.0f}")
    print(f"Labeled user-hours: {by_day['labeled_user_hours'].sum():,.0f}")
    display(by_day.head(20))

    ax = by_day.plot(
        x="relative_day_index",
        y="labeled_user_hours",
        figsize=(12, 4),
        legend=False,
    )
    ax.set(title="Red-team-labeled user-hours by relative day", xlabel="Relative day index", ylabel="Labeled user-hours")
    plt.tight_layout()
    plt.show()
else:
    print("Feature Parquet not found. Run `python -m ml.prepare_login_data` first.")

## Interpretation and limitations

- Use the manifest for full-file processing and label-match counts; do not generalize statistics from the first 100,000 authentication rows.
- The exact label match leaves some red-team labels unmatched. The preprocessing pipeline reports those rather than guessing a match.
- A labeled user-hour means at least one supplied red-team event was aggregated into that user's hour. Unlabeled hours are not guaranteed benign.
- The login detector is an initial baseline with sparse positives and low precision. See `evaluation/results/lanl_login_anomaly.json` for the evaluation and its caveats.
- To reproduce the complete feature generation and evaluation, run `python -m ml.prepare_login_data` and then `python -m ml.train_login_anomaly --n-estimators 100` from the repository root.